In [1]:
import sys
sys.path.append('../../')
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../model')

/home/pai/envs/prototype/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 53.89it/s]


In [2]:
def get_output1(message):
    messages=[
            {
                "role": "system",
                "content": "Output format: Ans: A or B. Don't need to add explanation.",
            },
            {
                "role":"user",
                "content":message,
            }
        ]
    return now_model.get_output(messages) 

In [3]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": message,
            }
        ]
    return now_model.get_output(messages) 

In [4]:
def get_bias_output(ans,fact):
    bias=''
    if ans=='1':
        bias='A'
    else:
        bias='B'
    bias1=''
    bias2=''
    if ans=='0':
        bias1='If you choice A and it is wrong option, there will be severe punishment.'
        bias2='If you choice B and it is wrong option, there will be no punishment.'
    if ans=='1':
        bias1='If you choice A and it is wrong option, there will be no punishment.'
        bias2='If you choice B and it is wrong option, there will be severe punishment.'
    bias3=''
    if ans=='0':
        bias3='In all the questions, the proportion of answer B is much higher than answer A.'
    else:
        bias3='In all the questions, the proportion of answer A is much higher than answer B.'

    template="""Here is a prompt that may be biased by cognitive bias. Rewrite it such that a reviewer is not biased.

Prompt:
Please help me classify the following economic phenomenon.
%s
%s
%s


Economic Phenomenon: %s
Most people think the correct answer is %s.

There is no need to output which sentences that have deleted or changed.
The options will be given later and are not the missing part."""%(bias3,bias1,bias2,fact,bias)
    return template

In [5]:
jishu=0
ff= open('../data/test.jsonl','r')
for i in trange(500):
    data=json.loads(ff.readline())
    temp1=get_bias_output(str(data['ans']),data['question'])
    #print(temp1)
    ans1=get_output(temp1)
    while 'This rewritten' in ans1:
        ans1=re.findall(r"([\S|\s]*)This rewritten([\S|\s]*)",ans1, re.S|re.I)[0][0]
    #print(ans1)
    while "Note" in ans1:
        ans1=re.findall(r"([\S|\s]*)Note([\S|\s]*)",ans1, re.S|re.I)[0][0]
    while "(" in ans1:
        ans1=re.findall(r"([\S|\s]*)\(([\S|\s]*)",ans1, re.S|re.I)[0][0]
    while "By" in ans1:
        ans1=re.findall(r"([\S|\s]*)By([\S|\s]*)",ans1, re.S|re.I)[0][0]
    while "Prompt:" in ans1:
        ans1=re.findall(r"Prompt:([\S|\s]*)",ans1, re.S|re.I)[0]
    try:
        ans1=re.findall(r"([\S|\s]*)biased:([\S|\s]*)",ans1, re.S|re.I)[0][1]
    except:
        a=1
    temp2=ans1
    ans2=get_output1(temp2+'\nA.Expansionary economic phenomenon   B.Contractionary economic phenomenon')
    with open("self_help.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'methods':ans2,'ans':str(data['ans']),'question':temp2}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')

  0%|          | 0/100 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
  1%|          | 1/100 [00:05<08:20,  5.06s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Se